In [1]:
import requests
import csv
import json
import time
import random

def fetch_bilibili_food_ranking():
    """
    抓取B站美食区排行榜数据
    API文档：https://api.bilibili.com/x/web-interface/ranking/v2?rid=211&type=all
    返回JSON格式的排行榜数据
    """
    
    # B站美食区排行榜API
    # 注意：rid参数控制分区编号，211代表美食区
    url = "https://api.bilibili.com/x/web-interface/ranking/v2?rid=211&type=all"
    
    # 设置请求头，模拟浏览器访问
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36',
        'Referer': 'https://www.bilibili.com/',
        'Accept': 'application/json, text/plain, */*'
    }
    
    try:
        # 发送GET请求获取数据
        response = requests.get(url, headers=headers, timeout=10)
        response.raise_for_status()  # 检查请求是否成功
        
        # 解析JSON响应数据
        data = response.json()
        
        # 检查API返回状态码
        if data['code'] != 0:
            print(f"API请求失败，错误码：{data['code']}, 错误信息：{data.get('message', '未知错误')}")
            return None
        
        # 提取视频列表数据
        video_list = data['data']['list']
        
        # 检查数据数量
        print(f"成功获取到 {len(video_list)} 条视频数据")
        
        return video_list
        
    except requests.exceptions.RequestException as e:
        print(f"网络请求失败: {e}")
        return None
    except json.JSONDecodeError as e:
        print(f"JSON解析失败: {e}")
        return None
    except KeyError as e:
        print(f"数据格式错误，缺少关键字段: {e}")
        return None

def parse_video_data(video_list):
    """
    解析视频数据，提取所需字段
    """
    parsed_data = []
    
    for video in video_list:
        try:
            # 提取核心字段
            video_info = {
                'aid': video.get('aid', ''),  # 视频ID
                'bvid': video.get('bvid', ''),  # BV号
                'title': video.get('title', '').replace('\n', ' ').replace(',', '，'),  # 视频标题，清理特殊字符
                # 注意：播放量数据在stat字段下的view键中
                '播放量': video.get('stat', {}).get('view', 0),
                # 注意：点赞数数据在stat字段下的like键中
                '点赞数': video.get('stat', {}).get('like', 0),
                # 注意：投币数数据在stat字段下的coin键中
                '投币数': video.get('stat', {}).get('coin', 0),
                # 注意：属地信息在pub_location字段中
                '属地': video.get('pub_location', '未知'),
                # 注意：视频时长数据在duration字段中，单位是秒
                '视频时长(秒)': video.get('duration', 0),
                # 转换为分钟格式
                '视频时长(分:秒)': f"{video.get('duration', 0)//60}:{video.get('duration', 0)%60:02d}",
                'up主': video.get('owner', {}).get('name', ''),
                'up主ID': video.get('owner', {}).get('mid', ''),
                '发布时间': time.strftime('%Y-%m-%d %H:%M:%S', time.localtime(video.get('pubdate', 0))) if video.get('pubdate') else '未知',
                '视频分类': video.get('tname', ''),
                '收藏数': video.get('stat', {}).get('favorite', 0),
                '分享数': video.get('stat', {}).get('share', 0),
                '评论数': video.get('stat', {}).get('reply', 0),
                '弹幕数': video.get('stat', {}).get('danmaku', 0)
            }
            parsed_data.append(video_info)
            
        except Exception as e:
            print(f"解析视频数据时出错 (aid: {video.get('aid', '未知')}): {e}")
            continue
    
    return parsed_data

def save_to_csv(data, filename="bilibili_food_ranking.csv"):
    """
    将数据保存到CSV文件
    """
    if not data:
        print("没有数据可保存")
        return False
    
    try:
        # 定义CSV文件的列顺序
        fieldnames = [
            'aid', 'bvid', 'title', '播放量', '点赞数', '投币数', 
            '属地', '视频时长(秒)', '视频时长(分:秒)', 'up主', 'up主ID',
            '发布时间', '视频分类', '收藏数', '分享数', '评论数', '弹幕数'
        ]
        
        with open(filename, 'w', newline='', encoding='utf-8-sig') as csvfile:
            writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
            writer.writeheader()
            writer.writerows(data)
        
        print(f"数据已成功保存到 {filename}")
        print(f"总计保存了 {len(data)} 条记录")
        return True
        
    except Exception as e:
        print(f"保存CSV文件时出错: {e}")
        return False

def get_random_samples(data, num_samples=3):
    """
    随机选择指定数量的视频作为样本
    """
    if len(data) < num_samples:
        print(f"数据量不足，只有 {len(data)} 条记录")
        return data
    
    return random.sample(data, num_samples)

def display_sample_info(samples):
    """
    显示随机样本信息
    """
    print("\n" + "="*60)
    print("随机选择的3个视频样本信息（用于数据验证）：")
    print("="*60)
    
    for i, sample in enumerate(samples, 1):
        print(f"\n【样本{i}】")
        print(f"视频标题：{sample['title'][:50]}...")
        print(f"BV号：{sample['bvid']}")
        print(f"播放量：{sample['播放量']:,}")
        print(f"点赞数：{sample['点赞数']:,}")
        print(f"投币数：{sample['投币数']:,}")
        print(f"属地：{sample['属地']}")
        print(f"视频时长：{sample['视频时长(分:秒)']}")
        print(f"视频链接：https://www.bilibili.com/video/{sample['bvid']}")
    
    print("\n" + "="*60)
    print("数据验证说明：")
    print("1. 使用上方提供的视频链接访问B站网页")
    print("2. 在视频页面核对播放量、点赞数、投币数、属地等信息")
    print("3. 确认抓取数据与网页显示数据是否一致")
    print("="*60)

def main():
    """
    主函数：执行完整的抓取、保存和验证流程
    """
    print("开始抓取B站美食区排行榜数据...")
    print("-" * 50)
    
    # 1. 抓取数据
    video_list = fetch_bilibili_food_ranking()
    if not video_list:
        print("数据抓取失败，程序终止")
        return
    
    # 2. 解析数据
    parsed_data = parse_video_data(video_list)
    if not parsed_data:
        print("数据解析失败，程序终止")
        return
    
    # 3. 保存数据到CSV
    csv_filename = "bilibili_food_ranking.csv"
    save_success = save_to_csv(parsed_data, csv_filename)
    
    if not save_success:
        print("数据保存失败")
        return
    
    # 4. 随机选择3个样本用于验证
    samples = get_random_samples(parsed_data, 3)
    
    # 5. 显示样本信息
    display_sample_info(samples)
    
    print(f"\n任务完成！所有数据已保存到 {csv_filename}")

if __name__ == "__main__":
    main()

开始抓取B站美食区排行榜数据...
--------------------------------------------------
成功获取到 100 条视频数据
数据已成功保存到 bilibili_food_ranking.csv
总计保存了 100 条记录

随机选择的3个视频样本信息（用于数据验证）：

【样本1】
视频标题：黄金比例烧麦，小当家惊呆了！！...
BV号：BV14DZJYXEEt
播放量：4,657,555
点赞数：186,190
投币数：6,624
属地：湖南
视频时长：5:38
视频链接：https://www.bilibili.com/video/BV14DZJYXEEt

【样本2】
视频标题：盘点一些小众，但味道不错的蔬菜，第18期～！...
BV号：BV1sRZ7YGEw4
播放量：287,218
点赞数：24,098
投币数：676
属地：山东
视频时长：2:00
视频链接：https://www.bilibili.com/video/BV1sRZ7YGEw4

【样本3】
视频标题：探索日本网红拉面店！超人气拉面馆，真的好吃吗？【大阪篇】...
BV号：BV14sZqYeEgc
播放量：807,001
点赞数：36,182
投币数：7,858
属地：日本
视频时长：10:39
视频链接：https://www.bilibili.com/video/BV14sZqYeEgc

数据验证说明：
1. 使用上方提供的视频链接访问B站网页
2. 在视频页面核对播放量、点赞数、投币数、属地等信息
3. 确认抓取数据与网页显示数据是否一致

任务完成！所有数据已保存到 bilibili_food_ranking.csv
